# 🧠 Train Virgo-1.0-Angkor 12B — Kaggle (free GPU)

A smarter Virgo: the same Virgo training, on **Gemma 3 12B** (3× bigger than 4B), in 4 bits (QLoRA) so it fits Kaggle's free T4. Answers, Khmer and reasoning get noticeably better.

**First time only**
1. On Hugging Face, open [google/gemma-3-12b-it](https://huggingface.co/google/gemma-3-12b-it) and click **Agree and access**.
2. Kaggle → **Settings → Phone verification**.
3. **Add-ons → Secrets**: `HF_TOKEN` (a Hugging Face token with **write** access), ticked for this notebook.

**Run**
1. **Session options**: **Accelerator → GPU T4 x2**, **Internet → On**.
2. **Save Version → Save & Run All (Commit)**. It runs in the background (about 2–4 hours), so you can close the tab.
3. It saves **Virgo-1.0-Angkor-12B** to your Hugging Face account (private). The server notebooks use it automatically from then on (the 4B stays as a backup).

Afterwards, open the version's log: it has Virgo's score and sample answers. Send them to Claude.

In [ ]:
# ⚙️ Settings
BASE_MODEL = "google/gemma-3-12b-it"
MODEL_NAME = "Virgo-1.0-Angkor-12B"
EPOCHS = 3
OUT = "chat/out/virgo-1.0-angkor-12b"

In [ ]:
# 1️⃣ Set up
import os, subprocess, sys, torch
assert torch.cuda.is_available(), "❌ No GPU: Session options → Accelerator → GPU T4 x2"
print("✅ GPUs:", [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())])
WORK = "/kaggle/working"
os.chdir(WORK)
if not os.path.isdir("virgo-models/.git"):
    r = subprocess.run(["git", "clone", "-q", "https://github.com/nangmrr752/virgo-models"])
    assert r.returncode == 0, "❌ Can't download: Session options → Internet → On (needs phone verification)"
os.chdir(f"{WORK}/virgo-models")
subprocess.run(["git", "pull", "-q"])
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements-train.txt"], check=True)
subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "-q", "torchao"])  # conflicts with newer peft
from huggingface_hub import login, HfApi
try:
    from kaggle_secrets import UserSecretsClient
    token = UserSecretsClient().get_secret("HF_TOKEN")
except Exception:
    token = None
assert token, "❌ Add the HF_TOKEN secret (Add-ons → Secrets) and tick it for this notebook"
login(token=token)
REPO = f"{HfApi().whoami()['name']}/{MODEL_NAME}"
print("✅ Ready. Virgo 12B will be saved to", REPO)

def run(cmd):
    """Runs a step and shows its output here; on failure, shows the last lines (send them to Claude)."""
    import subprocess, collections
    tail = collections.deque(maxlen=60)
    proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in proc.stdout:
        print(line, end="")
        tail.append(line)
    if proc.wait() != 0:
        print("\n" + "=" * 60 + "\n❌ It stopped with an error. Send these last lines to Claude:\n" + "=" * 60)
        print("".join(tail))
        raise SystemExit(f"❌ {next((a for a in cmd if a.endswith('.py')), cmd[0])} failed (exit code {proc.returncode})")


In [ ]:
# 2️⃣ Train Virgo on Gemma 3 12B
import subprocess, sys
run([sys.executable, "scripts/check_data.py"])
run([sys.executable, "-u", "chat/train.py", "--base", BASE_MODEL, "--epochs", str(EPOCHS), "--out", OUT])

In [ ]:
# 3️⃣ Save Virgo-1.0-Angkor-12B to your Hugging Face account (private)
from huggingface_hub import HfApi
card = f"""---
base_model: {BASE_MODEL}
library_name: peft
license: gemma
language: [en, km]
tags: [virgo, ksn, khmer, lora]
---
# {MODEL_NAME}

**{MODEL_NAME}** is KSN's own assistant model for [Virgo AI](https://virgoai.camksn.com): friendly, clear and honest, in English and Khmer.

It is a LoRA fine-tune of Google's [{BASE_MODEL}](https://huggingface.co/{BASE_MODEL}), trained on Virgo's own conversations.
Use is subject to the [Gemma Terms of Use](https://ai.google.dev/gemma/terms).

Made by KSN · https://soknang.camksn.com/
"""
open(f"{OUT}/README.md", "w", encoding="utf-8").write(card)
api = HfApi()
api.create_repo(REPO, private=True, exist_ok=True)
api.upload_folder(folder_path=OUT, repo_id=REPO, commit_message=MODEL_NAME, ignore_patterns=["checkpoint-*"])
print(f"✅ Saved {MODEL_NAME}: https://huggingface.co/{REPO}")

In [ ]:
# 4️⃣ Score Virgo 12B (send this table to Claude)
import subprocess, sys
run([sys.executable, "chat/evaluate.py", "--adapter", OUT])

In [ ]:
# 5️⃣ Try Virgo-1.0-Angkor 12B
import sys; sys.path.append("chat")
from virgo_chat import VirgoChat
bot = VirgoChat(adapter=OUT)
for q in ["Who are you?", "អ្នកជានរណា?", "What is 15% of 200?", "ពន្យល់ AI ឱ្យងាយយល់", "Translate to Khmer: Thank you for your help."]:
    print("🧑", q)
    print("🤖", bot.reply([{"role": "user", "content": q}], temperature=0), "\n")
print("🎉 Done! Send the score table and these answers to Claude.")